# Notebook 1: การดึงข้อมูลจากเอกสารด้วย Docling
# (Document Extraction with Docling)

## สิ่งที่จะได้เรียนรู้ (What you'll learn)
- ติดตั้งและใช้งาน Docling สำหรับแปลงเอกสาร
- แปลงไฟล์ PDF และ PPTX ภาษาไทย
- สำรวจโครงสร้าง DoclingDocument
- ดึงตาราง รูปภาพ และลำดับชั้นของเอกสาร
- บันทึกผลลัพธ์เป็น Markdown และ JSON

In [13]:
# ติดตั้ง dependencies (Install dependencies)
%pip install docling pandas tabulate

# แก้ปัญหา symlink บน Windows (Fix Windows symlink issue for HuggingFace model downloads)
import os, sys
if sys.platform == "win32":
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
    import huggingface_hub.file_download as _hf_dl, shutil
    _orig = _hf_dl._create_symlink
    def _copy_fallback(src, dst, new_blob=False):
        try:
            _orig(src, dst, new_blob=new_blob)
        except OSError:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.exists(dst):
                os.remove(dst)
            shutil.copy2(src, dst)
    _hf_dl._create_symlink = _copy_fallback
    print("Applied Windows symlink workaround")
else:
    print("Unix system — no workaround needed")

Unix system — no workaround needed


In [14]:
# ตรวจสอบการติดตั้ง (Verify installation)
from importlib.metadata import version
print(f"Docling version: {version('docling')}")

from docling.document_converter import DocumentConverter
print("DocumentConverter imported successfully!")

Docling version: 2.78.0
DocumentConverter imported successfully!


## DocumentConverter คืออะไร?

`DocumentConverter` เป็นตัวแปลงเอกสารหลักของ Docling ที่รองรับหลายรูปแบบ:
- **PDF** — รวมถึง scanned PDF ที่ต้องใช้ OCR
- **PPTX** — ไฟล์ PowerPoint
- **DOCX** — ไฟล์ Word
- **HTML, Images, และอื่นๆ**

ผลลัพธ์จะได้เป็น `DoclingDocument` ซึ่งเป็นโครงสร้างข้อมูลแบบรวม (unified representation)

In [15]:
from docling.document_converter import DocumentConverter
from pathlib import Path

# กำหนดไฟล์ต้นทาง (Set source file)
pdf_path = Path("data/thai_sample.pdf")

# สร้าง converter และแปลงเอกสาร
converter = DocumentConverter()
result = converter.convert(str(pdf_path))

print(f"สถานะการแปลง: {result.status}")
print(f"ชื่อไฟล์: {result.input.file.name}")

[INFO] 2026-03-12 02:28:30,872 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-12 02:28:30,872 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-12 02:28:30,885 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-12 02:28:30,886 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-12 02:28:31,061 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-12 02:28:31,062 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-12 02:28:31,065 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-12 02:28:31,066 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-12 02:28:31,141 [RapidO

สถานะการแปลง: ConversionStatus.SUCCESS
ชื่อไฟล์: thai_sample.pdf


In [16]:
# ดูผลลัพธ์เป็น Markdown
markdown_text = result.document.export_to_markdown()

print("=" * 60)
print("Markdown Output (500 ตัวอักษรแรก):")
print("=" * 60)
print(markdown_text[:500])
print(f"\n... (ทั้งหมด {len(markdown_text)} ตัวอักษร)")

Markdown Output (500 ตัวอักษรแรก):
## บทที่  ปัญญาประดิษฐ์คืออะไร

ปัญญาประดิษฐ์

หรือ

คือสาขาวิชาทางวิทยาการคอมพิวเตอร์

ที่มุ่งเน้นการสร้างระบบที่สามารถทำงานที่ต้องอาศัยความฉลาดของมนุษย์ เช่น การเรียนรู้ การตัดสินใจ การรู้จำภาพ การเข้าใจภาษา และการแก้ปัญหา

## สามารถแบ่งออกเป็นหลายประเภท ได้แก่

แบบแคบ   - ออกแบบมาเพื่อทำงานเฉพาะอย่าง เช่น การแปลภาษา แบบทั่วไป   - สามารถทำงานได้หลากหลายเหมือนมนุษย์ ขั้นสูง   - มีความสามารถเหนือกว่ามนุษย์ในทุกด้าน

## บทที่  การเรียนรู้ของเครื่อง

การเรียนรู้ของเครื่อง เป็นสาขาย่อยของปัญญาประดิ

... (ทั้งหมด 1825 ตัวอักษร)


In [17]:
import json

# ดูโครงสร้าง JSON
json_doc = result.document.export_to_dict()

# แสดง keys ระดับบนสุด
print("Keys ในเอกสาร:")
for key in json_doc.keys():
    print(f"  - {key}")

# บันทึกเป็นไฟล์
output_path = Path("output/extracted")
output_path.mkdir(parents=True, exist_ok=True)

with open(output_path / f"{pdf_path.stem}.json", "w", encoding="utf-8") as f:
    json.dump(json_doc, f, ensure_ascii=False, indent=2)

print(f"\nบันทึก JSON ไปที่: {output_path / f'{pdf_path.stem}.json'}")

Keys ในเอกสาร:
  - schema_name
  - version
  - name
  - origin
  - furniture
  - body
  - groups
  - texts
  - pictures
  - tables
  - key_value_items
  - form_items
  - pages

บันทึก JSON ไปที่: output/extracted/thai_sample.json


In [18]:
# สำรวจโครงสร้างของเอกสาร (Document hierarchy)
doc = result.document

# นับจำนวน elements
print("สรุปโครงสร้างเอกสาร:")
print(f"  จำนวนตาราง (Tables): {len(list(doc.tables))}")
print(f"  จำนวนรูปภาพ (Pictures): {len(list(doc.pictures))}")

# แสดง text elements
print("\nเนื้อหาข้อความ (แสดง 5 รายการแรก):")
for i, (item, _level) in enumerate(doc.iterate_items()):
    if i >= 5:
        print("  ...")
        break
    text_preview = item.text[:80] if hasattr(item, 'text') else str(type(item).__name__)
    print(f"  [{i}] {text_preview}")

สรุปโครงสร้างเอกสาร:
  จำนวนตาราง (Tables): 0
  จำนวนรูปภาพ (Pictures): 0

เนื้อหาข้อความ (แสดง 5 รายการแรก):
  [0] บทที่  ปัญญาประดิษฐ์คืออะไร
  [1] ปัญญาประดิษฐ์
  [2] หรือ
  [3] คือสาขาวิชาทางวิทยาการคอมพิวเตอร์
  [4] ที่มุ่งเน้นการสร้างระบบที่สามารถทำงานที่ต้องอาศัยความฉลาดของมนุษย์ เช่น การเรียน
  ...


In [19]:
import pandas as pd

# ดึงตารางจากเอกสาร
tables = list(doc.tables)
print(f"พบตาราง {len(tables)} ตาราง\n")

for i, table in enumerate(tables):
    print(f"--- ตาราง {i+1} ---")
    df = table.export_to_dataframe(doc=doc)
    print(df.to_markdown(index=False))

    # บันทึกเป็น CSV
    csv_path = output_path / f"{pdf_path.stem}_table_{i+1}.csv"
    df.to_csv(csv_path, index=False)
    print(f"บันทึกที่: {csv_path}\n")

พบตาราง 0 ตาราง



In [20]:
# แปลงไฟล์ PowerPoint
pptx_path = Path("datathai_sample.pptx")

if pptx_path.exists():
    pptx_result = converter.convert(str(pptx_path))
    pptx_markdown = pptx_result.document.export_to_markdown()

    print("PPTX Markdown Output (500 ตัวอักษรแรก):")
    print("=" * 60)
    print(pptx_markdown[:500])

    # บันทึก
    with open(output_path / f"{pptx_path.stem}.json", "w", encoding="utf-8") as f:
        json.dump(pptx_result.document.export_to_dict(), f, ensure_ascii=False, indent=2)

    # บันทึก Markdown
    with open(output_path / f"{pptx_path.stem}.md", "w", encoding="utf-8") as f:
        f.write(pptx_markdown)

    print(f"\nบันทึกผลลัพธ์ไปที่ {output_path}/")
else:
    print(f"ไม่พบไฟล์ {pptx_path} — ข้ามขั้นตอนนี้")

ไม่พบไฟล์ datathai_sample.pptx — ข้ามขั้นตอนนี้


In [21]:
from pathlib import Path

# แปลงไฟล์ทั้งหมดในโฟลเดอร์ data/samples/
sample_dir = Path("data")
all_files = list(sample_dir.glob("*.pdf")) + list(sample_dir.glob("*.pptx"))

print(f"พบไฟล์ {len(all_files)} ไฟล์:")
for f in all_files:
    print(f"  - {f.name}")

# แปลงทีละไฟล์
results = []
for file_path in all_files:
    print(f"\nกำลังแปลง: {file_path.name}...")
    res = converter.convert(str(file_path))
    results.append(res)

    # บันทึก Markdown
    md_path = output_path / f"{file_path.stem}.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(res.document.export_to_markdown())

    # บันทึก JSON
    json_path = output_path / f"{file_path.stem}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(res.document.export_to_dict(), f, ensure_ascii=False, indent=2)

    print(f"  สถานะ: {res.status}")
    print(f"  บันทึก: {md_path.name}, {json_path.name}")

print(f"\nแปลงเสร็จสิ้น {len(results)} ไฟล์!")

[INFO] 2026-03-12 02:28:39,708 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-12 02:28:39,712 [RapidOCR] device_config.py:57: Using CPU device


พบไฟล์ 2 ไฟล์:
  - thai_sample.pdf
  - thai_sample.pptx

กำลังแปลง: thai_sample.pdf...


[INFO] 2026-03-12 02:28:39,736 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-12 02:28:39,737 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-12 02:28:40,069 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-12 02:28:40,070 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-12 02:28:40,072 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-12 02:28:40,073 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-12 02:28:40,199 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-12 02:28:40,200 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-12 02:28:40,223 [RapidO

  สถานะ: ConversionStatus.SUCCESS
  บันทึก: thai_sample.md, thai_sample.json

กำลังแปลง: thai_sample.pptx...
  สถานะ: ConversionStatus.SUCCESS
  บันทึก: thai_sample.md, thai_sample.json

แปลงเสร็จสิ้น 2 ไฟล์!


## บันทึกผลลัพธ์สำหรับ Notebook ถัดไป

เราบันทึกข้อมูลที่ดึงออกมาไว้ใน `output/extracted/`:
- ไฟล์ `.md` — Markdown สำหรับอ่านง่าย
- ไฟล์ `.json` — JSON สำหรับประมวลผลต่อ
- ไฟล์ `.csv` — ตารางที่ดึงออกมา

ใน Notebook 2 เราจะนำข้อมูลเหล่านี้มาทำความสะอาดและแบ่งเป็นชิ้นส่วน (chunks)

In [22]:
import os

# สรุปไฟล์ที่สร้าง
print("ไฟล์ที่สร้างใน output/extracted/:")
for f in sorted(output_path.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name} ({size:,} bytes)")

ไฟล์ที่สร้างใน output/extracted/:
  thai_sample.json (59,077 bytes)
  thai_sample.md (6,642 bytes)
